<a href="https://colab.research.google.com/github/SANGHATI23/genomic-evidence-reliability/blob/main/17_GES_Aware_Genomic_RAG_Cell_7C10_Hybrid_Single_Reviewer_Evaluation_Packet_Materialization_V2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not running in Google Colab; Drive mount skipped.')

ROOT = Path('/content/drive/MyDrive/GES_RAG_Temporal_Study')
if not ROOT.exists():
    raise FileNotFoundError(
        f'Project root not found: {ROOT}\n'
        'Confirm Google Drive is mounted and the project directory is unchanged.'
    )

print(f'Project root: {ROOT}')

Mounted at /content/drive
Project root: /content/drive/MyDrive/GES_RAG_Temporal_Study


## 1. Imports, exact Cell 7C9 lineage, and output paths

In [2]:
from __future__ import annotations

from collections import OrderedDict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any
import csv
import hashlib
import json
import re
import tempfile

import numpy as np
import pandas as pd
import pyarrow.parquet as pq


NOTEBOOK_NAME = (
    '17_GES_Aware_Genomic_RAG_Cell_7C10_'
    'Hybrid_Single_Reviewer_Evaluation_Packet_Materialization.ipynb'
)
CELL_ID = '7C10'
STAGE = '7C'
AMENDMENT_ID = 'A003'
PACKAGE_VERSION = 'v1'
CREATED_UTC = datetime.now(timezone.utc).isoformat()

EXPECTED_RESPONSES = 1_440
EXPECTED_QUESTIONS = 80
EXPECTED_RESPONSES_PER_QUESTION = 18
EXPECTED_RUBRICS_PER_RESPONSE = 8
EXPECTED_FIRSTPASS_RUBRIC_ROWS = 11_520
EXPECTED_REPEAT_ITEMS = 160
EXPECTED_REPEAT_ITEMS_PER_QUESTION = 2
EXPECTED_REPEAT_RUBRIC_ROWS = 1_280
MINIMUM_WASHOUT_DAYS = 14
SINGLE_REVIEWER_ID = 'SINGLE-BLINDED-REVIEWER'

EXPECTED_CELL_7C9_TERMINAL_DECISION = (
    'PASS_STAGE7C9_PROTOCOL_AMENDMENT_A003_HYBRID_SINGLE_REVIEWER_EVALUATION_'
    'FROZEN_BEFORE_HUMAN_SCORING_DETERMINISTIC_ALL1440_SINGLE_BLINDED_HUMAN_'
    'REVIEW_ALL1440_AND_160_ITEM_14DAY_INTRARATER_REPEAT_PRESPECIFIED_CELL7C10_'
    'PACKET_MATERIALIZATION_ONLY_AUTHORIZED_NO_CONDITION_UNBLINDING_RUN_AGGREGATION_'
    'PRIMARY_ENDPOINT_CALCULATION_BOOTSTRAP_OR_ARM_COMPARISON'
)

EXPECTED_CELL_7C9_AUTHORIZATION_DECISION = (
    'AUTHORIZE_STAGE7C_CELL7C10_HYBRID_SINGLE_REVIEWER_EVALUATION_PACKET_'
    'MATERIALIZATION_ONLY_FROM_FROZEN_CELL7C8_PACKAGE_WITH_DETERMINISTIC_SCORING_'
    'SPEC_SINGLE_BLINDED_REVIEWER_ALL_1440_RESPONSES_AND_160_RESPONSE_14DAY_'
    'INTRARATER_REPEAT_SAMPLE_NO_CONDITION_UNBLINDING_ARM_COMPARISON_RUN_AGGREGATION_'
    'PRIMARY_ENDPOINT_CALCULATION_OR_BOOTSTRAP'
)

# --------------------------------------------------------------------------------------------------
# Exact Cell 7C9 / A003 package from the successful terminal PASS.
# --------------------------------------------------------------------------------------------------
CELL_7C9_CONFIG_DIR = (
    ROOT / 'configs' / 'stage7_rag'
    / 'protocol_amendment_A003_hybrid_single_reviewer_evaluation_v1'
)
CELL_7C9_QC_DIR = (
    ROOT / 'outputs' / 'quality_checks' / 'stage7_rag'
    / 'protocol_amendment_A003_hybrid_single_reviewer_evaluation_v1'
)

CELL_7C9 = OrderedDict([
    ('amendment', {
        'path': CELL_7C9_CONFIG_DIR / 'protocol_amendment_A003_hybrid_single_reviewer_evaluation_v1.json',
        'sha256': '558dfe320a398f31c18786a965171b2c777c3f329f73763c5010791944b5a943',
    }),
    ('deterministic_scoring_spec', {
        'path': CELL_7C9_CONFIG_DIR / 'protocol_amendment_A003_deterministic_scoring_spec_v1.json',
        'sha256': '8bda7feaa34e0d127269787906030585e0fdea30b3d6893f3b03ba54ab2b9cc4',
    }),
    ('human_review_spec', {
        'path': CELL_7C9_CONFIG_DIR / 'protocol_amendment_A003_single_blinded_human_review_spec_v1.json',
        'sha256': '239638bef6c572dbc1ff05cac32ccc0f69f7da3fd25e748475602639949a249a',
    }),
    ('repeat_assessment_spec', {
        'path': CELL_7C9_CONFIG_DIR / 'protocol_amendment_A003_intrarater_repeat_assessment_spec_v1.json',
        'sha256': 'cef2b0721c345ad5407917ca8879fa39a7e99a7cfa915e048af5c8e5c2572558',
    }),
    ('input_inventory', {
        'path': CELL_7C9_CONFIG_DIR / 'protocol_amendment_A003_verified_input_inventory_v1.csv',
        'sha256': '2e6c7cd24ad74f871d06815ac0eb1298d6ca461a2b9d663c05c4e0a457b2d4a3',
    }),
    ('qc', {
        'path': CELL_7C9_QC_DIR / 'protocol_amendment_A003_qc_v1.json',
        'sha256': '5acbf6f544b2262e594cf2c76270e8ed56c5ed3c9f0bee29ee15b52b89187fac',
    }),
    ('manifest', {
        'path': CELL_7C9_CONFIG_DIR / 'protocol_amendment_A003_manifest_v1.json',
        'sha256': '9b75ab9fc6d91ba588b2e95d5469406f7aa9353eedc529deacd9f8f96076cc3a',
    }),
])

# --------------------------------------------------------------------------------------------------
# Exact Cell 7C8 authorized inputs.
# --------------------------------------------------------------------------------------------------
CELL_7C8_EXEC_DIR = (
    ROOT / 'outputs' / 'rag_execution' / 'stage7_rag'
    / 'cell_7c8_blinded_reviewer_packet_v1'
)
CELL_7C8_CONFIG_DIR = (
    ROOT / 'configs' / 'stage7_rag'
    / 'cell_7c8_blinded_reviewer_packet_v1'
)

CELL_7C8_REVIEW_PACKET = {
    'path': CELL_7C8_EXEC_DIR / 'cell_7c8_blinded_review_packet_v1.parquet',
    'sha256': '232ea13fb2e09fa8821bc61fa8c4bd9d7f04f8c30eb6b47360ed8ee1f49d8466',
}
CELL_7C8_ROUTING = {
    'path': CELL_7C8_CONFIG_DIR / 'cell_7c8_internal_blinded_review_routing_map_v1.parquet',
    'sha256': '8c65375e6a24761bd14e2d59d837507c67146ed88e8e64a533bca304598c696c',
}
CELL_7C8_MANIFEST = {
    'path': CELL_7C8_CONFIG_DIR / 'cell_7c8_blinded_reviewer_packet_manifest_v1.json',
    'sha256': 'e8a2bf79d7c0479063af0fbfb73a77427746c79bdc11600672a82af678b25af7',
}

# --------------------------------------------------------------------------------------------------
# Cell 7C10 output package.
# --------------------------------------------------------------------------------------------------
EXEC_DIR = (
    ROOT / 'outputs' / 'rag_execution' / 'stage7_rag'
    / 'cell_7c10_hybrid_single_reviewer_evaluation_packet_v1'
)
CONFIG_DIR = (
    ROOT / 'configs' / 'stage7_rag'
    / 'cell_7c10_hybrid_single_reviewer_evaluation_packet_v1'
)
QC_DIR = (
    ROOT / 'outputs' / 'quality_checks' / 'stage7_rag'
    / 'cell_7c10_hybrid_single_reviewer_evaluation_packet_v1'
)

OUTPUTS = OrderedDict([
    ('firstpass_review_packet',
     EXEC_DIR / 'cell_7c10_firstpass_single_reviewer_packet_v1.parquet'),
    ('firstpass_assignment',
     EXEC_DIR / 'cell_7c10_firstpass_single_reviewer_assignment_v1.csv'),
    ('firstpass_rubric_template',
     EXEC_DIR / 'cell_7c10_firstpass_rubric_scoring_template_v1.csv'),
    ('firstpass_atomic_claim_template',
     EXEC_DIR / 'cell_7c10_firstpass_atomic_claim_annotation_template_v1.csv'),
    ('deterministic_scoring_input',
     EXEC_DIR / 'cell_7c10_deterministic_scoring_input_v1.parquet'),
    ('repeat_sample_inventory',
     CONFIG_DIR / 'cell_7c10_repeat_sample_inventory_internal_v1.csv'),
    ('repeat_review_packet',
     EXEC_DIR / 'cell_7c10_repeat_reblinded_review_packet_LOCKED_v1.parquet'),
    ('repeat_rubric_template',
     EXEC_DIR / 'cell_7c10_repeat_rubric_scoring_template_LOCKED_v1.csv'),
    ('repeat_atomic_claim_template',
     EXEC_DIR / 'cell_7c10_repeat_atomic_claim_annotation_template_LOCKED_v1.csv'),
    ('repeat_internal_routing_map',
     CONFIG_DIR / 'cell_7c10_repeat_internal_routing_map_v1.parquet'),
    ('repeat_release_gate',
     CONFIG_DIR / 'cell_7c10_repeat_release_gate_v1.json'),
    ('reviewer_instructions',
     CONFIG_DIR / 'cell_7c10_single_reviewer_instructions_v1.json'),
    ('input_inventory',
     CONFIG_DIR / 'cell_7c10_verified_input_inventory_v1.csv'),
    ('execution_report',
     QC_DIR / 'cell_7c10_packet_materialization_execution_report_v1.json'),
    ('qc',
     QC_DIR / 'cell_7c10_packet_materialization_qc_v1.json'),
    ('manifest',
     CONFIG_DIR / 'cell_7c10_hybrid_single_reviewer_packet_manifest_v1.json'),
])

for directory in (EXEC_DIR, CONFIG_DIR, QC_DIR):
    directory.mkdir(parents=True, exist_ok=True)

existing = [str(path) for path in OUTPUTS.values() if path.exists()]
if existing:
    raise FileExistsError(
        'Cell 7C10 fail-closed overwrite protection is active. Existing output(s):\\n- '
        + '\\n- '.join(existing)
    )

print(f'Execution directory: {EXEC_DIR}')
print(f'Config directory   : {CONFIG_DIR}')
print(f'QC directory       : {QC_DIR}')

Execution directory: /content/drive/MyDrive/GES_RAG_Temporal_Study/outputs/rag_execution/stage7_rag/cell_7c10_hybrid_single_reviewer_evaluation_packet_v1
Config directory   : /content/drive/MyDrive/GES_RAG_Temporal_Study/configs/stage7_rag/cell_7c10_hybrid_single_reviewer_evaluation_packet_v1
QC directory       : /content/drive/MyDrive/GES_RAG_Temporal_Study/outputs/quality_checks/stage7_rag/cell_7c10_hybrid_single_reviewer_evaluation_packet_v1


## 2. Checksum, serialization, and canonical-JSON helpers

In [3]:
def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        while True:
            block = handle.read(chunk_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()


def sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode('utf-8')).hexdigest()


def sidecar_path(path: Path) -> Path:
    return path.with_name(path.name + '.sha256')


def read_sidecar_hash(path: Path) -> str:
    text = path.read_text(encoding='utf-8').strip()
    if not text:
        raise ValueError(f'Empty sidecar: {path}')
    token = text.split()[0].strip()
    if not re.fullmatch(r'[0-9a-fA-F]{64}', token):
        raise ValueError(f'Invalid sidecar format: {path}')
    return token.lower()


def sidecar_is_valid(path: Path) -> bool:
    return (
        path.exists()
        and sidecar_path(path).exists()
        and read_sidecar_hash(sidecar_path(path)) == sha256_file(path)
    )


def verify_exact_artifact(label: str, path: Path, expected_sha256: str) -> dict[str, Any]:
    if not path.exists():
        raise FileNotFoundError(f'Missing frozen artifact [{label}]: {path}')
    observed = sha256_file(path)
    if observed != expected_sha256:
        raise AssertionError(
            f'{label} SHA-256 mismatch.\\nExpected: {expected_sha256}\\nObserved: {observed}'
        )
    if not sidecar_is_valid(path):
        raise AssertionError(f'Invalid/missing sidecar for {label}: {path}')
    return {
        'input_id': label,
        'path': str(path),
        'sha256': observed,
        'bytes': int(path.stat().st_size),
        'sidecar_path': str(sidecar_path(path)),
        'sidecar_valid': True,
    }


def load_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding='utf-8'))


def to_native(value: Any) -> Any:
    if value is None or isinstance(value, (str, int, float, bool)):
        return value
    if isinstance(value, np.generic):
        return to_native(value.item())
    if isinstance(value, np.ndarray):
        return [to_native(v) for v in value.tolist()]
    if isinstance(value, (list, tuple)):
        return [to_native(v) for v in value]
    if isinstance(value, dict):
        return {str(k): to_native(v) for k, v in value.items()}
    try:
        if pd.isna(value):
            return None
    except Exception:
        pass
    return str(value)


def canonical_json(payload: Any) -> str:
    return json.dumps(
        to_native(payload),
        ensure_ascii=False,
        sort_keys=True,
        separators=(',', ':'),
        allow_nan=False,
    )


def stable_write_json(path: Path, payload: Any) -> str:
    path.write_text(
        json.dumps(
            to_native(payload),
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            allow_nan=False,
        ) + chr(10),
        encoding='utf-8',
    )
    return sha256_file(path)


def stable_write_csv(path: Path, frame: pd.DataFrame) -> str:
    frame.to_csv(path, index=False, encoding='utf-8', lineterminator=chr(10))
    return sha256_file(path)


def stable_write_parquet(path: Path, frame: pd.DataFrame) -> str:
    frame.to_parquet(path, index=False, engine='pyarrow', compression='zstd')
    return sha256_file(path)


def write_sidecar(path: Path) -> None:
    sidecar_path(path).write_text(
        f'{sha256_file(path)}  {path.name}' + chr(10),
        encoding='utf-8',
    )


with tempfile.TemporaryDirectory(prefix='cell_7c10_writer_test_') as tmp:
    p = Path(tmp) / 'x.json'
    stable_write_json(p, {'ok': True})
    write_sidecar(p)
    assert load_json(p) == {'ok': True}
    assert sidecar_is_valid(p)

print('Serialization / SHA-256 helper self-test: PASS')

Serialization / SHA-256 helper self-test: PASS


## 3. Reverify Cell 7C9 / A003 authorization and Cell 7C8 inputs

In [4]:
verified_inputs = []

for artifact_id, spec in CELL_7C9.items():
    record = verify_exact_artifact(
        f'cell_7c9_{artifact_id}',
        spec['path'],
        spec['sha256'],
    )
    record['source_cell'] = '7C9'
    verified_inputs.append(record)

for label, spec in [
    ('cell_7c8_review_packet', CELL_7C8_REVIEW_PACKET),
    ('cell_7c8_internal_routing_map', CELL_7C8_ROUTING),
    ('cell_7c8_manifest', CELL_7C8_MANIFEST),
]:
    record = verify_exact_artifact(label, spec['path'], spec['sha256'])
    record['source_cell'] = '7C8'
    verified_inputs.append(record)

amendment_7c9 = load_json(CELL_7C9['amendment']['path'])
manifest_7c9 = load_json(CELL_7C9['manifest']['path'])
qc_7c9 = load_json(CELL_7C9['qc']['path'])
det_spec = load_json(CELL_7C9['deterministic_scoring_spec']['path'])
human_spec = load_json(CELL_7C9['human_review_spec']['path'])
repeat_spec = load_json(CELL_7C9['repeat_assessment_spec']['path'])

if manifest_7c9.get('terminal_decision') != EXPECTED_CELL_7C9_TERMINAL_DECISION:
    raise AssertionError('Cell 7C9 terminal PASS mismatch.')
if amendment_7c9.get('authorization_decision') != EXPECTED_CELL_7C9_AUTHORIZATION_DECISION:
    raise AssertionError('Cell 7C9 Cell 7C10 authorization decision mismatch.')
if manifest_7c9.get('next_authorized_cell') != '7C10':
    raise AssertionError('Cell 7C9 does not authorize Cell 7C10.')
if manifest_7c9.get('condition_unblinding_authorized') is not False:
    raise AssertionError('Cell 7C9 unexpectedly authorizes condition unblinding.')
if manifest_7c9.get('run_aggregation_authorized') is not False:
    raise AssertionError('Cell 7C9 unexpectedly authorizes run aggregation.')
if manifest_7c9.get('primary_endpoint_calculation_authorized') is not False:
    raise AssertionError('Cell 7C9 unexpectedly authorizes primary endpoint calculation.')
if manifest_7c9.get('bootstrap_inference_authorized') is not False:
    raise AssertionError('Cell 7C9 unexpectedly authorizes bootstrap inference.')
if int(qc_7c9.get('failed_checks', -1)) != 0:
    raise AssertionError('Cell 7C9 QC does not report zero failures.')

if human_spec['review_population']['reviewer_count'] != 1:
    raise AssertionError('A003 human-review spec no longer freezes one reviewer.')
if human_spec['review_population']['response_count'] != 1_440:
    raise AssertionError('A003 human-review population changed.')
if repeat_spec['sampling']['total_repeat_items'] != 160:
    raise AssertionError('A003 repeat sample size changed.')
if repeat_spec['sampling']['items_per_question'] != 2:
    raise AssertionError('A003 repeat items-per-question changed.')
if repeat_spec['timing']['minimum_washout_days_after_first_pass_completion'] != 14:
    raise AssertionError('A003 washout interval changed.')

print('Cell 7C9 / A003 package               : 7/7 exact hashes + sidecars')
print('Cell 7C8 authorized inputs             : 3/3 exact hashes + sidecars')
print('Cell 7C10 authorization                : VERIFIED')
print('Single reviewer                        : FROZEN')
print('Repeat sample                          : 160 = 2 per question')
print('Minimum washout                        : 14 days')
print('Condition unblinding                   : PROHIBITED')

Cell 7C9 / A003 package               : 7/7 exact hashes + sidecars
Cell 7C8 authorized inputs             : 3/3 exact hashes + sidecars
Cell 7C10 authorization                : VERIFIED
Single reviewer                        : FROZEN
Repeat sample                          : 160 = 2 per question
Minimum washout                        : 14 days
Condition unblinding                   : PROHIBITED


## 4. Load the score-blind Cell 7C8 packet and validate its invariants

In [5]:
review_packet = pd.read_parquet(CELL_7C8_REVIEW_PACKET['path'])
internal_routing = pd.read_parquet(CELL_7C8_ROUTING['path'])

REQUIRED_REVIEW_COLUMNS = {
    'review_item_id',
    'question_id',
    'question_text',
    'model_response_json',
    'structured_answer_key_json',
    'rubric_bundle_json',
    'score_blind_context_bundle_json',
}
missing = sorted(REQUIRED_REVIEW_COLUMNS - set(review_packet.columns))
if missing:
    raise AssertionError('Cell 7C8 review packet missing required columns: ' + ', '.join(missing))

if len(review_packet) != EXPECTED_RESPONSES:
    raise AssertionError(f'Expected 1,440 review items; observed {len(review_packet):,}.')
if review_packet['review_item_id'].duplicated().any():
    raise AssertionError('Duplicate review_item_id detected.')

question_counts = review_packet.groupby('question_id', dropna=False).size()
if len(question_counts) != EXPECTED_QUESTIONS:
    raise AssertionError(f'Expected 80 questions; observed {len(question_counts)}.')
if not question_counts.eq(EXPECTED_RESPONSES_PER_QUESTION).all():
    raise AssertionError('Every question must have exactly 18 frozen responses.')

# Reviewer-facing packet must remain free of routing/condition identifiers.
for prohibited in [
    'blinded_alias',
    'run_id',
    'generation_request_id',
    'prompt_instance_id',
    'condition_id',
    'condition_name',
]:
    if prohibited in review_packet.columns:
        raise AssertionError(f'Reviewer packet exposes prohibited column: {prohibited}')

# Internal routing is needed only for future reconstruction, not human review or repeat selection.
required_routing = {
    'review_item_id',
    'generation_request_id',
    'prompt_instance_id',
    'question_id',
    'blinded_alias',
    'run_id',
}
missing_routing = sorted(required_routing - set(internal_routing.columns))
if missing_routing:
    raise AssertionError(
        'Cell 7C8 internal routing map missing required fields: ' + ', '.join(missing_routing)
    )

if len(internal_routing) != 1_440:
    raise AssertionError('Cell 7C8 routing map must contain 1,440 rows.')
if set(review_packet['review_item_id']) != set(internal_routing['review_item_id']):
    raise AssertionError('Review packet and internal routing review_item_id sets differ.')

print(f'Blinded review items                   : {len(review_packet):,}')
print(f'Primary questions                      : {question_counts.size}')
print('Frozen responses per question          : 18')
print('Reviewer-facing condition identifiers  : NONE')
print('Repeat selection will use routing map  : NO')

Blinded review items                   : 1,440
Primary questions                      : 80
Frozen responses per question          : 18
Reviewer-facing condition identifiers  : NONE
Repeat selection will use routing map  : NO


## 5. Materialize the single-reviewer first-pass package

In [6]:
# Deterministic score-blind first-pass order.
firstpass_assignment = review_packet[['review_item_id', 'question_id']].copy()
firstpass_assignment['_order_hash'] = firstpass_assignment['review_item_id'].map(
    lambda value: sha256_text(f'A003_FIRSTPASS|{value}')
)
firstpass_assignment = firstpass_assignment.sort_values(
    ['_order_hash', 'review_item_id'],
    kind='mergesort',
).reset_index(drop=True)
firstpass_assignment['review_order'] = np.arange(
    1, len(firstpass_assignment) + 1, dtype=np.int32
)
firstpass_assignment['reviewer_id'] = SINGLE_REVIEWER_ID
firstpass_assignment = firstpass_assignment[
    ['reviewer_id', 'review_order', 'review_item_id', 'question_id']
]

# Reviewer packet in first-pass review order.
firstpass_review_packet = firstpass_assignment.merge(
    review_packet,
    on=['review_item_id', 'question_id'],
    how='left',
    validate='one_to_one',
).sort_values('review_order', kind='mergesort').reset_index(drop=True)

# One rubric row per frozen rubric dimension.
rubric_rows = []
for row in firstpass_review_packet.itertuples(index=False):
    rubric_bundle = json.loads(row.rubric_bundle_json)
    if not isinstance(rubric_bundle, list) or len(rubric_bundle) != 8:
        raise AssertionError(
            f'Review item {row.review_item_id} does not contain exactly 8 rubric assignments.'
        )
    for rubric_index, rubric_assignment in enumerate(rubric_bundle, start=1):
        rubric_rows.append({
            'reviewer_id': SINGLE_REVIEWER_ID,
            'review_order': int(row.review_order),
            'review_item_id': str(row.review_item_id),
            'question_id': str(row.question_id),
            'rubric_index': rubric_index,
            'rubric_assignment_json': canonical_json(rubric_assignment),
            'reviewer_rubric_score': '',
            'reviewer_rubric_notes': '',
            'review_complete': '',
        })

firstpass_rubric_template = pd.DataFrame(rubric_rows)

# One row per response for atomic-claim decomposition and judgments.
firstpass_atomic_claim_template = firstpass_assignment.copy()
firstpass_atomic_claim_template['atomic_claim_annotations_json'] = ''
firstpass_atomic_claim_template['overall_reviewer_notes'] = ''
firstpass_atomic_claim_template['review_complete'] = ''

if len(firstpass_review_packet) != 1_440:
    raise AssertionError('First-pass packet must contain 1,440 rows.')
if len(firstpass_assignment) != 1_440:
    raise AssertionError('First-pass assignment must contain 1,440 rows.')
if len(firstpass_rubric_template) != EXPECTED_FIRSTPASS_RUBRIC_ROWS:
    raise AssertionError(
        f'Expected 11,520 first-pass rubric rows; observed {len(firstpass_rubric_template):,}.'
    )
if len(firstpass_atomic_claim_template) != 1_440:
    raise AssertionError('First-pass atomic-claim template must contain 1,440 rows.')

print(f'First-pass review items                : {len(firstpass_review_packet):,}')
print(f'First-pass reviewer assignments        : {len(firstpass_assignment):,}')
print(f'First-pass rubric rows                 : {len(firstpass_rubric_template):,}')
print(f'First-pass atomic-claim rows           : {len(firstpass_atomic_claim_template):,}')
print('Reviewer identity                      : SINGLE-BLINDED-REVIEWER')

First-pass review items                : 1,440
First-pass reviewer assignments        : 1,440
First-pass rubric rows                 : 11,520
First-pass atomic-claim rows           : 1,440
Reviewer identity                      : SINGLE-BLINDED-REVIEWER


## 6. Materialize deterministic-scoring inputs — no scoring performed

In [7]:
deterministic_rows = []

for row in review_packet.itertuples(index=False):
    response_payload = json.loads(row.model_response_json)
    answer_key_payload = json.loads(row.structured_answer_key_json)
    context_bundle = json.loads(row.score_blind_context_bundle_json)

    if not isinstance(context_bundle, list) or len(context_bundle) != 5:
        raise AssertionError(
            f'Review item {row.review_item_id} does not contain exactly five context blocks.'
        )

    evidence_ids = response_payload.get('evidence_ids', [])
    if not isinstance(evidence_ids, list):
        raise AssertionError(f'evidence_ids is not a list for {row.review_item_id}.')

    context_packet_ids = [
        str(item['packet_id'])
        for item in context_bundle
    ]

    deterministic_rows.append({
        'review_item_id': str(row.review_item_id),
        'question_id': str(row.question_id),
        'response_policy': str(response_payload.get('response_policy', '')),
        'conflict_detected': response_payload.get('conflict_detected'),
        'evidence_strength': str(response_payload.get('evidence_strength', '')),
        'confidence': response_payload.get('confidence'),
        'cited_evidence_ids_json': canonical_json(evidence_ids),
        'context_packet_ids_json': canonical_json(context_packet_ids),
        'expected_abstention_or_qualification_required':
            answer_key_payload.get('expected_abstention_or_qualification_required'),
        'deterministic_scores_calculated': False,
    })

deterministic_scoring_input = pd.DataFrame(deterministic_rows)

if len(deterministic_scoring_input) != 1_440:
    raise AssertionError('Deterministic-scoring input must contain 1,440 rows.')
if not deterministic_scoring_input['deterministic_scores_calculated'].eq(False).all():
    raise AssertionError('Cell 7C10 must not calculate deterministic scores.')

print(f'Deterministic-scoring input rows       : {len(deterministic_scoring_input):,}')
print('Evidence-ID membership scored          : NO')
print('Required-caution compliance scored     : NO')
print('Primary endpoint scored                : NO')

Deterministic-scoring input rows       : 1,440
Evidence-ID membership scored          : NO
Required-caution compliance scored     : NO
Primary endpoint scored                : NO


## 7. Freeze the exact 160-item repeat sample and new re-blinded IDs

In [8]:
repeat_selection_rows = []

for question_id, group in review_packet.groupby('question_id', sort=False):
    candidate = group[['review_item_id', 'question_id']].copy()
    candidate['_selection_hash'] = candidate['review_item_id'].map(
        lambda value: sha256_text(
            f'A003_REPEAT|{question_id}|{value}'
        )
    )
    candidate = candidate.sort_values(
        ['_selection_hash', 'review_item_id'],
        kind='mergesort',
    ).reset_index(drop=True)

    selected = candidate.iloc[:EXPECTED_REPEAT_ITEMS_PER_QUESTION].copy()
    if len(selected) != 2:
        raise AssertionError(f'Question {question_id} did not yield exactly 2 repeat items.')

    for selection_rank, (_, selected_row) in enumerate(
        selected.iterrows(),
        start=1,
    ):
        original_review_item_id = str(selected_row['review_item_id'])
        selection_hash = str(selected_row['_selection_hash'])
        repeat_item_id = 'RPT-' + sha256_text(
            f'A003_REBLIND|{original_review_item_id}'
        )[:24].upper()

        repeat_selection_rows.append({
            'question_id': str(question_id),
            'selection_rank_within_question': selection_rank,
            'selection_hash_sha256': selection_hash,
            'review_item_id': original_review_item_id,
            'repeat_item_id': repeat_item_id,
        })

repeat_internal = pd.DataFrame(repeat_selection_rows).sort_values(
    ['question_id', 'selection_rank_within_question'],
    kind='mergesort',
).reset_index(drop=True)

if len(repeat_internal) != EXPECTED_REPEAT_ITEMS:
    raise AssertionError(f'Expected 160 repeat items; observed {len(repeat_internal)}.')
if repeat_internal['repeat_item_id'].duplicated().any():
    raise AssertionError('Repeat re-blinded ID collision detected.')
if not repeat_internal.groupby('question_id').size().eq(2).all():
    raise AssertionError('Repeat sample is not exactly two items per question.')

# Reviewer-facing repeat sample inventory removes original review_item_id and selection hash.
repeat_sample_inventory = repeat_internal[
    ['repeat_item_id', 'question_id', 'selection_rank_within_question']
].copy()

# Build re-blinded repeat packet.
repeat_review_packet = repeat_internal[
    ['repeat_item_id', 'review_item_id', 'question_id']
].merge(
    review_packet,
    on=['review_item_id', 'question_id'],
    how='left',
    validate='one_to_one',
)

repeat_review_packet = repeat_review_packet.drop(
    columns=['review_item_id']
)

repeat_review_packet['_repeat_order_hash'] = repeat_review_packet['repeat_item_id'].map(
    lambda value: sha256_text(f'A003_REPEAT_ORDER|{value}')
)
repeat_review_packet = repeat_review_packet.sort_values(
    ['_repeat_order_hash', 'repeat_item_id'],
    kind='mergesort',
).reset_index(drop=True)
repeat_review_packet['repeat_order'] = np.arange(
    1, len(repeat_review_packet) + 1, dtype=np.int32
)
repeat_review_packet = repeat_review_packet.drop(columns=['_repeat_order_hash'])

# Repeat rubric and claim templates use repeat IDs only.
repeat_rubric_rows = []
for row in repeat_review_packet.itertuples(index=False):
    rubric_bundle = json.loads(row.rubric_bundle_json)
    if not isinstance(rubric_bundle, list) or len(rubric_bundle) != 8:
        raise AssertionError(
            f'Repeat item {row.repeat_item_id} does not contain exactly 8 rubric assignments.'
        )
    for rubric_index, rubric_assignment in enumerate(rubric_bundle, start=1):
        repeat_rubric_rows.append({
            'reviewer_id': SINGLE_REVIEWER_ID,
            'repeat_order': int(row.repeat_order),
            'repeat_item_id': str(row.repeat_item_id),
            'question_id': str(row.question_id),
            'rubric_index': rubric_index,
            'rubric_assignment_json': canonical_json(rubric_assignment),
            'reviewer_rubric_score': '',
            'reviewer_rubric_notes': '',
            'review_complete': '',
        })

repeat_rubric_template = pd.DataFrame(repeat_rubric_rows)

repeat_atomic_claim_template = repeat_review_packet[
    ['repeat_order', 'repeat_item_id', 'question_id']
].copy()
repeat_atomic_claim_template.insert(0, 'reviewer_id', SINGLE_REVIEWER_ID)
repeat_atomic_claim_template['atomic_claim_annotations_json'] = ''
repeat_atomic_claim_template['overall_reviewer_notes'] = ''
repeat_atomic_claim_template['review_complete'] = ''

# Internal mapping is not reviewer-facing.
repeat_internal_routing_map = repeat_internal.merge(
    internal_routing[
        [
            'review_item_id',
            'generation_request_id',
            'prompt_instance_id',
            'question_id',
            'blinded_alias',
            'run_id',
        ]
    ],
    on=['review_item_id', 'question_id'],
    how='left',
    validate='one_to_one',
)

if len(repeat_review_packet) != 160:
    raise AssertionError('Repeat review packet must contain 160 rows.')
if len(repeat_rubric_template) != EXPECTED_REPEAT_RUBRIC_ROWS:
    raise AssertionError(
        f'Expected 1,280 repeat rubric rows; observed {len(repeat_rubric_template)}.'
    )
if len(repeat_atomic_claim_template) != 160:
    raise AssertionError('Repeat atomic-claim template must contain 160 rows.')

# Strong repeat reviewer-facing blinding.
for frame_name, frame in {
    'repeat_review_packet': repeat_review_packet,
    'repeat_sample_inventory': repeat_sample_inventory,
    'repeat_rubric_template': repeat_rubric_template,
    'repeat_atomic_claim_template': repeat_atomic_claim_template,
}.items():
    for prohibited_column in [
        'review_item_id',
        'blinded_alias',
        'run_id',
        'generation_request_id',
        'prompt_instance_id',
        'condition_id',
        'condition_name',
    ]:
        if prohibited_column in frame.columns:
            raise AssertionError(
                f'{frame_name} exposes prohibited repeat-review field: {prohibited_column}'
            )

print(f'Repeat items                           : {len(repeat_review_packet)}')
print('Repeat sampling                        : exactly 2 per each of 80 questions')
print('Repeat ID namespace                    : RPT-...')
print('Original review_item_id in repeat file : NO')
print('Condition / alias / run ID             : HIDDEN')

Repeat items                           : 160
Repeat sampling                        : exactly 2 per each of 80 questions
Repeat ID namespace                    : RPT-...
Original review_item_id in repeat file : NO
Condition / alias / run ID             : HIDDEN


## 8. Freeze the 14-day repeat-release gate and reviewer instructions

In [9]:
repeat_release_gate = {
    'protocol_amendment_id': AMENDMENT_ID,
    'cell_id': CELL_ID,
    'created_utc': CREATED_UTC,
    'repeat_packet_status': 'LOCKED',
    'minimum_washout_days': MINIMUM_WASHOUT_DAYS,
    'washout_anchor':
        'The UTC completion timestamp of the frozen first-pass human-review package, '
        'to be recorded by a later first-pass completion/freeze cell.',
    'release_rule':
        'Repeat assessment may begin only when current_utc >= first_pass_completion_utc + 14 days.',
    'first_pass_completion_utc': None,
    'repeat_eligible_not_before_utc': None,
    'first_pass_scores_must_be_hidden_during_repeat': True,
    'first_pass_notes_must_be_hidden_during_repeat': True,
    'original_review_item_id_must_be_hidden_during_repeat': True,
    'condition_identity_must_remain_hidden': True,
    'current_release_authorized': False,
    'note':
        'The repeat files are materialized now solely to freeze sample membership and re-blinded IDs. '
        'Do not open or use reviewer-facing repeat files until a later release authorization verifies '
        'first-pass completion plus the 14-day minimum washout.',
}

reviewer_instructions = {
    'protocol_amendment_id': AMENDMENT_ID,
    'cell_id': CELL_ID,
    'created_utc': CREATED_UTC,
    'reviewer_id': SINGLE_REVIEWER_ID,
    'first_pass': {
        'review_all_1440_items': True,
        'use_packet': OUTPUTS['firstpass_review_packet'].name,
        'use_assignment': OUTPUTS['firstpass_assignment'].name,
        'use_rubric_template': OUTPUTS['firstpass_rubric_template'].name,
        'use_atomic_claim_template': OUTPUTS['firstpass_atomic_claim_template'].name,
        'atomic_claim_annotation_schema': [
            {
                'claim_index': 1,
                'claim_text': 'verbatim atomic factual claim',
                'correctness': 'correct|incorrect|uncertain',
                'citation_support': 'supported|unsupported|no_citation|uncertain',
                'cited_evidence_ids': ['packet_id_or_empty'],
                'notes': 'optional',
            }
        ],
        'do_not_access_internal_routing_map': True,
        'do_not_attempt_condition_inference': True,
    },
    'repeat_assessment': {
        'sample_size': 160,
        'minimum_washout_days': 14,
        'release_currently_authorized': False,
        'repeat_packet_filename': OUTPUTS['repeat_review_packet'].name,
        'first_pass_scores_visible': False,
        'first_pass_notes_visible': False,
        'original_review_item_id_visible': False,
    },
    'primary_endpoint': {
        'definition':
            'fraction of atomic factual claims that are both correct and citation-supported',
        'calculate_during_cell_7c10': False,
        'calculate_during_manual_review': False,
        'calculation_requires_later_frozen-review authorization': True,
    },
    'blinding': {
        'condition_identity_hidden': True,
        'blinded_alias_hidden': True,
        'run_id_hidden': True,
        'GES_scores_hidden': True,
        'quality_and_retrieval_ranks_hidden': True,
    },
}

print('Repeat release status                    : LOCKED')
print('Washout anchor                           : future first-pass completion UTC')
print('Minimum washout                          : 14 days')
print('Repeat files usable now                  : NO')
print('First-pass human review usable after PASS: YES')

Repeat release status                    : LOCKED
Washout anchor                           : future first-pass completion UTC
Minimum washout                          : 14 days
Repeat files usable now                  : NO
First-pass human review usable after PASS: YES


## 9. Freeze the Cell 7C10 package

In [10]:
prewrite_checks = OrderedDict([
    ('firstpass_packet_1440', len(firstpass_review_packet) == 1440),
    ('firstpass_assignment_1440', len(firstpass_assignment) == 1440),
    ('firstpass_unique_items', firstpass_review_packet['review_item_id'].nunique() == 1440),
    ('firstpass_rubric_11520', len(firstpass_rubric_template) == 11520),
    ('firstpass_claim_1440', len(firstpass_atomic_claim_template) == 1440),
    ('deterministic_input_1440', len(deterministic_scoring_input) == 1440),
    ('deterministic_scores_not_calculated',
     deterministic_scoring_input['deterministic_scores_calculated'].eq(False).all()),
    ('repeat_items_160', len(repeat_review_packet) == 160),
    ('repeat_two_per_question',
     repeat_sample_inventory.groupby('question_id').size().eq(2).all()),
    ('repeat_rubric_1280', len(repeat_rubric_template) == 1280),
    ('repeat_claim_160', len(repeat_atomic_claim_template) == 160),
    ('repeat_ids_unique', repeat_review_packet['repeat_item_id'].nunique() == 160),
    ('repeat_original_ids_hidden',
     'review_item_id' not in repeat_review_packet.columns),
    ('repeat_alias_hidden',
     'blinded_alias' not in repeat_review_packet.columns),
    ('repeat_run_hidden',
     'run_id' not in repeat_review_packet.columns),
    ('repeat_release_locked',
     repeat_release_gate['current_release_authorized'] is False),
    ('washout_14_days',
     repeat_release_gate['minimum_washout_days'] == 14),
    ('condition_unblinding_not_performed', True),
    ('run_aggregation_not_performed', True),
    ('primary_endpoint_not_calculated', True),
    ('bootstrap_not_performed', True),
    ('arm_comparison_not_performed', True),
    ('llm_not_called', True),
])

failed = [name for name, passed in prewrite_checks.items() if not bool(passed)]
if failed:
    raise RuntimeError(
        'Cell 7C10 prewrite QC failed:\\n- ' + '\\n- '.join(failed)
    )

stable_write_parquet(
    OUTPUTS['firstpass_review_packet'], firstpass_review_packet
); write_sidecar(OUTPUTS['firstpass_review_packet'])

stable_write_csv(
    OUTPUTS['firstpass_assignment'], firstpass_assignment
); write_sidecar(OUTPUTS['firstpass_assignment'])

stable_write_csv(
    OUTPUTS['firstpass_rubric_template'], firstpass_rubric_template
); write_sidecar(OUTPUTS['firstpass_rubric_template'])

stable_write_csv(
    OUTPUTS['firstpass_atomic_claim_template'], firstpass_atomic_claim_template
); write_sidecar(OUTPUTS['firstpass_atomic_claim_template'])

stable_write_parquet(
    OUTPUTS['deterministic_scoring_input'], deterministic_scoring_input
); write_sidecar(OUTPUTS['deterministic_scoring_input'])

stable_write_csv(
    OUTPUTS['repeat_sample_inventory'], repeat_sample_inventory
); write_sidecar(OUTPUTS['repeat_sample_inventory'])

stable_write_parquet(
    OUTPUTS['repeat_review_packet'], repeat_review_packet
); write_sidecar(OUTPUTS['repeat_review_packet'])

stable_write_csv(
    OUTPUTS['repeat_rubric_template'], repeat_rubric_template
); write_sidecar(OUTPUTS['repeat_rubric_template'])

stable_write_csv(
    OUTPUTS['repeat_atomic_claim_template'], repeat_atomic_claim_template
); write_sidecar(OUTPUTS['repeat_atomic_claim_template'])

stable_write_parquet(
    OUTPUTS['repeat_internal_routing_map'], repeat_internal_routing_map
); write_sidecar(OUTPUTS['repeat_internal_routing_map'])

stable_write_json(
    OUTPUTS['repeat_release_gate'], repeat_release_gate
); write_sidecar(OUTPUTS['repeat_release_gate'])

stable_write_json(
    OUTPUTS['reviewer_instructions'], reviewer_instructions
); write_sidecar(OUTPUTS['reviewer_instructions'])

input_inventory = pd.DataFrame(verified_inputs)
stable_write_csv(OUTPUTS['input_inventory'], input_inventory)
write_sidecar(OUTPUTS['input_inventory'])

terminal_decision = (
    'PASS_STAGE7C10_A003_HYBRID_SINGLE_REVIEWER_PACKET_MATERIALIZED_'
    'FIRSTPASS1440_RUBRIC11520_ATOMICCLAIM1440_DETERMINISTIC_INPUT1440_'
    'REPEAT160_REBLINDED_TWO_PER_QUESTION_REPEAT_RUBRIC1280_14DAY_RELEASE_GATE_'
    'LOCKED_CHECKSUM_PROTECTED_FIRSTPASS_HUMAN_REVIEW_MAY_BEGIN_NO_CONDITION_'
    'UNBLINDING_RUN_AGGREGATION_PRIMARY_ENDPOINT_BOOTSTRAP_OR_ARM_COMPARISON_'
    'NEXT_AUTOMATED_EXECUTION_NOT_AUTHORIZED'
)

execution_report = {
    'cell_id': CELL_ID,
    'stage': STAGE,
    'protocol_amendment_id': AMENDMENT_ID,
    'package_version': PACKAGE_VERSION,
    'created_utc': CREATED_UTC,
    'notebook': NOTEBOOK_NAME,
    'authorization': {
        'cell_7c9_manifest_sha256': CELL_7C9['manifest']['sha256'],
        'cell_7c9_authorization_decision': EXPECTED_CELL_7C9_AUTHORIZATION_DECISION,
    },
    'materialized_counts': {
        'firstpass_review_items': 1440,
        'firstpass_assignments': 1440,
        'firstpass_rubric_rows': 11520,
        'firstpass_atomic_claim_rows': 1440,
        'deterministic_scoring_input_rows': 1440,
        'repeat_items': 160,
        'repeat_items_per_question': 2,
        'repeat_rubric_rows': 1280,
        'repeat_atomic_claim_rows': 160,
    },
    'repeat_gate': {
        'minimum_washout_days': 14,
        'currently_released': False,
        'release_requires_future_firstpass_completion_freeze': True,
    },
    'scientific_operations': {
        'human_scoring_performed': False,
        'deterministic_scoring_performed': False,
        'condition_identity_unblinded': False,
        'run_aggregation_performed': False,
        'primary_endpoint_calculated': False,
        'bootstrap_inference_performed': False,
        'arm_comparison_performed': False,
        'llm_called': False,
    },
    'next_required_action':
        'Complete the first-pass single blinded human review using only the Cell 7C10 first-pass reviewer-facing files. '
        'Do not use repeat files yet. After first-pass completion, freeze/import the completed review and record the '
        'completion timestamp before any repeat-assessment release.',
    'terminal_decision': terminal_decision,
}

stable_write_json(OUTPUTS['execution_report'], execution_report)
write_sidecar(OUTPUTS['execution_report'])

qc_payload = {
    'cell_id': CELL_ID,
    'stage': STAGE,
    'protocol_amendment_id': AMENDMENT_ID,
    'package_version': PACKAGE_VERSION,
    'created_utc': CREATED_UTC,
    'checks': {name: bool(value) for name, value in prewrite_checks.items()},
    'passed_checks': len(prewrite_checks),
    'failed_checks': 0,
    'total_checks': len(prewrite_checks),
    'terminal_decision': terminal_decision,
}
stable_write_json(OUTPUTS['qc'], qc_payload)
write_sidecar(OUTPUTS['qc'])

manifest_payload = {
    'cell_id': CELL_ID,
    'stage': STAGE,
    'protocol_amendment_id': AMENDMENT_ID,
    'package_version': PACKAGE_VERSION,
    'created_utc': CREATED_UTC,
    'notebook': NOTEBOOK_NAME,
    'upstream_lineage': {
        'cell_7c9_manifest_sha256': CELL_7C9['manifest']['sha256'],
        'cell_7c8_review_packet_sha256': CELL_7C8_REVIEW_PACKET['sha256'],
        'cell_7c8_internal_routing_map_sha256': CELL_7C8_ROUTING['sha256'],
    },
    'output_artifacts': {
        key: {
            'path': str(path),
            'sha256': sha256_file(path),
            'sidecar_valid': sidecar_is_valid(path),
            'reviewer_facing_firstpass': key in {
                'firstpass_review_packet',
                'firstpass_assignment',
                'firstpass_rubric_template',
                'firstpass_atomic_claim_template',
                'reviewer_instructions',
            },
            'repeat_locked': key in {
                'repeat_review_packet',
                'repeat_rubric_template',
                'repeat_atomic_claim_template',
            },
        }
        for key, path in OUTPUTS.items()
        if key != 'manifest'
    },
    'firstpass_human_review_authorized_after_pass': True,
    'repeat_review_currently_authorized': False,
    'condition_unblinding_authorized': False,
    'run_aggregation_authorized': False,
    'primary_endpoint_calculation_authorized': False,
    'bootstrap_inference_authorized': False,
    'arm_comparison_authorized': False,
    'next_authorized_cell': None,
    'next_required_action':
        'Human first-pass blinded review; later separate first-pass completion/import freeze.',
    'terminal_decision': terminal_decision,
}

stable_write_json(OUTPUTS['manifest'], manifest_payload)
write_sidecar(OUTPUTS['manifest'])

# Fresh readback.
for path in OUTPUTS.values():
    if not path.exists() or not sidecar_is_valid(path):
        raise AssertionError(f'Cell 7C10 final readback failed: {path}')

rb_firstpass = pd.read_parquet(OUTPUTS['firstpass_review_packet'])
rb_firstassign = pd.read_csv(OUTPUTS['firstpass_assignment'])
rb_firstrubric = pd.read_csv(OUTPUTS['firstpass_rubric_template'])
rb_firstclaim = pd.read_csv(OUTPUTS['firstpass_atomic_claim_template'])
rb_det = pd.read_parquet(OUTPUTS['deterministic_scoring_input'])
rb_repeat = pd.read_parquet(OUTPUTS['repeat_review_packet'])
rb_repeat_rubric = pd.read_csv(OUTPUTS['repeat_rubric_template'])
rb_repeat_claim = pd.read_csv(OUTPUTS['repeat_atomic_claim_template'])
rb_gate = load_json(OUTPUTS['repeat_release_gate'])
rb_manifest = load_json(OUTPUTS['manifest'])
rb_qc = load_json(OUTPUTS['qc'])

readback_checks = OrderedDict([
    ('firstpass_1440', len(rb_firstpass) == 1440),
    ('assignment_1440', len(rb_firstassign) == 1440),
    ('first_rubric_11520', len(rb_firstrubric) == 11520),
    ('first_claim_1440', len(rb_firstclaim) == 1440),
    ('det_input_1440', len(rb_det) == 1440),
    ('repeat_160', len(rb_repeat) == 160),
    ('repeat_rubric_1280', len(rb_repeat_rubric) == 1280),
    ('repeat_claim_160', len(rb_repeat_claim) == 160),
    ('repeat_original_id_hidden', 'review_item_id' not in rb_repeat.columns),
    ('repeat_alias_hidden', 'blinded_alias' not in rb_repeat.columns),
    ('repeat_gate_locked', rb_gate['current_release_authorized'] is False),
    ('repeat_gate_14_days', rb_gate['minimum_washout_days'] == 14),
    ('manifest_next_none', rb_manifest.get('next_authorized_cell') is None),
    ('manifest_unblinding_false',
     rb_manifest.get('condition_unblinding_authorized') is False),
    ('manifest_primary_endpoint_false',
     rb_manifest.get('primary_endpoint_calculation_authorized') is False),
    ('qc_zero_failures', int(rb_qc.get('failed_checks', -1)) == 0),
    ('all_sidecars_valid', all(sidecar_is_valid(path) for path in OUTPUTS.values())),
])

failed_rb = [name for name, passed in readback_checks.items() if not bool(passed)]
if failed_rb:
    raise RuntimeError(
        'Cell 7C10 final readback QC failed:\\n- ' + '\\n- '.join(failed_rb)
    )

total_checks = len(prewrite_checks) + len(readback_checks)

separator = '=' * 158
print('\\n' + separator)
print('EXPERIMENT 2 — STAGE 7C — CELL 7C10')
print('A003 HYBRID SINGLE-REVIEWER EVALUATION PACKET MATERIALIZATION')
print(separator)
print(f'Notebook                                      : {NOTEBOOK_NAME}')
print(f'Project root                                  : {ROOT}')

print('\\nUPSTREAM A003 AUTHORIZATION')
print(f'Cell 7C9 manifest SHA-256                     : {CELL_7C9["manifest"]["sha256"]}')
print('Cell 7C9 terminal PASS verified               : YES')
print('Cell 7C10 packet authorization                : VERIFIED')

print('\\nFIRST-PASS SINGLE BLINDED REVIEW')
print(f'Review items                                  : {len(rb_firstpass):,}')
print(f'Assignments                                   : {len(rb_firstassign):,}')
print(f'Rubric scoring rows                           : {len(rb_firstrubric):,}')
print(f'Atomic-claim annotation rows                  : {len(rb_firstclaim):,}')
print(f'Reviewer                                      : {SINGLE_REVIEWER_ID}')
print('First-pass human review may begin             : YES')

print('\\nDETERMINISTIC EVALUATION')
print(f'Input rows                                    : {len(rb_det):,}')
print('Evidence-ID validity calculated              : NO')
print('Required-caution compliance calculated        : NO')
print('Scientific performance calculated            : NO')

print('\\nINTRA-RATER REPEAT FREEZE')
print(f'Frozen repeat sample                          : {len(rb_repeat):,}')
print('Sampling                                     : 2 opaque items per each of 80 questions')
print(f'Repeat rubric rows                            : {len(rb_repeat_rubric):,}')
print(f'Repeat atomic-claim rows                      : {len(rb_repeat_claim):,}')
print('New repeat IDs                               : YES — RPT-...')
print('Original review IDs in repeat packet          : NO')
print('Minimum washout                              : 14 days after first-pass completion')
print('Repeat assessment currently released          : NO — LOCKED')

print('\\nBLINDING / ANALYSIS BOUNDARY')
print('Condition identity unblinded                  : NO')
print('Frozen blinded alias visible to reviewer      : NO')
print('Run ID visible to reviewer                    : NO')
print('GES / quality / RRF scores                    : NO')
print('Run aggregation                               : NO')
print('Primary endpoint calculation                  : NO')
print('Bootstrap / arm comparison                    : NO')

print('\\nCELL 7C10 FROZEN OUTPUTS')
for label, path in OUTPUTS.items():
    print(f'{label:<46}: {path}')
    print(f'{"SHA-256":<46}: {sha256_file(path)}')

print(f'\\nQC checks                                      : {total_checks}/{total_checks} PASS')

print('\\nNEXT BOUNDARY')
print('First-pass human blinded review               : AUTHORIZED')
print('Repeat review                                 : NOT YET AUTHORIZED')
print('Next automated cell                           : NOT AUTHORIZED')
print('After first-pass completion                   : separate completion/import freeze + timestamp')
print('Condition unblinding / comparative analysis   : STILL PROHIBITED')

print(f'\\nFINAL DECISION                                : {terminal_decision}')
print(separator)

\n==============================================================================================================================================================
EXPERIMENT 2 — STAGE 7C — CELL 7C10
A003 HYBRID SINGLE-REVIEWER EVALUATION PACKET MATERIALIZATION
Notebook                                      : 17_GES_Aware_Genomic_RAG_Cell_7C10_Hybrid_Single_Reviewer_Evaluation_Packet_Materialization.ipynb
Project root                                  : /content/drive/MyDrive/GES_RAG_Temporal_Study
\nUPSTREAM A003 AUTHORIZATION
Cell 7C9 manifest SHA-256                     : 9b75ab9fc6d91ba588b2e95d5469406f7aa9353eedc529deacd9f8f96076cc3a
Cell 7C9 terminal PASS verified               : YES
Cell 7C10 packet authorization                : VERIFIED
\nFIRST-PASS SINGLE BLINDED REVIEW
Review items                                  : 1,440
Assignments                                   : 1,440
Rubric scoring rows                           : 11,520
Atomic-claim annotation rows                  : 1,4